### all important impotrs

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import requests
import gradio as gr

In [2]:
# Load environment variables from .env file
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

### Creating Tool

In [3]:
def get_weather(city):
    return {"weather": f"Weather in {city}: Sunny (mock data)"}

def save_lead(email):
    print("New lead:", email)
    return {"status": "lead saved"}

### Tools Definition here

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_lead",
            "description": "Save user's email",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string"}
                },
                "required": ["email"]
            }
        }
    }
]

### Tool Runner

In [5]:
tool_map = {
    "get_weather": get_weather,
    "save_lead": save_lead
}

def run_tools(tool_calls):
    results = []

    for call in tool_calls:
        tool_name = call.function.name
        args = json.loads(call.function.arguments)

        tool = tool_map.get(tool_name)

        if tool:
            result = tool(**args)
        else:
            result = {"error": "tool not found"}

        results.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(result)
        })

    return results

### Agent Loop (core)

In [6]:
def agent_loop(message, history):

    messages = [
        {"role": "system", "content": "You are a helpful AI agent."}
    ]

    messages.extend(history)
    messages.append({"role": "user", "content": message})

    while True:

        response = client.chat.completions.create(
            model="openai/gpt-4.1-mini",
            messages=messages,
            tools=tools
        )

        msg = response.choices[0].message

        if response.choices[0].finish_reason == "tool_calls":
            messages.append(msg)
            tool_results = run_tools(msg.tool_calls)
            messages.extend(tool_results)
        else:
            return msg.content

In [ ]:
## Gradio UI

In [9]:
with gr.Blocks() as demo:
    gr.Markdown("# Autonomous AI Agent")
    gr.Markdown("Agent can use tools and reason.")

    chatbot = gr.ChatInterface(agent_loop)

demo.launch()

/Users/sirphiltechpaxe/projects/agents/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/sirphiltechpaxe/projects/agents/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sirphiltechpaxe/projects/agents/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sirphiltechpaxe/projects/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sirphiltechpaxe/projects/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sirphiltechpaxe/projects/agents/.venv/lib/python3.12/site-pac